In [ ]:
import pandas as pd
import networkx as nx

# Load your CSV
#df = pd.read_csv("/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/iid220.ind.8.0.KNC.intra.IBD.16cM.frac0.7.date.csv")
df = pd.read_csv("/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC_v12_16cM_fracgp0.7.csv")

# Create an undirected graph
G = nx.Graph()

# Add edges to the graph with the weight as the 'sum' column
for _, row in df.iterrows():
    G.add_edge(row['iid1'], row['iid2'], weight=row['sum_IBD>8'])

# Calculate weighted centrality using degree centrality (you can choose other centrality measures)
centrality = nx.degree_centrality(G)

# If you want to consider the weight of the edges in centrality computation:
weighted_centrality = nx.betweenness_centrality(G, weight='weight')

# Show the weighted centrality
print(centrality)


In [ ]:
import pandas as pd
import networkx as nx

# Load your CSV
df = pd.read_csv("/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC_v12_16cM_fracgp0.7.csv")

# Create an undirected graph
G = nx.Graph()

# Add edges to the graph with the weight as the 'sum' column
for _, row in df.iterrows():
    G.add_edge(row['iid1'], row['iid2'], weight=row['sum_IBD>8'])

# Calculate the weighted strength of each node (sum of edge weights incident to the node)
strength = {}
for node in G.nodes():
    strength[node] = sum(d['weight'] for _, d in G[node].items())

# Show the strength for each node
print(strength)


In [ ]:
import pandas as pd
import networkx as nx

# Load your CSV
df = pd.read_csv("/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/v12/KNC_v12_16cM_fracgp0.7.csv")

# Create an undirected graph
G = nx.Graph()

# Add edges to the graph with the weight as the 'sum' column
for _, row in df.iterrows():
    G.add_edge(row['iid1'], row['iid2'], weight=row['sum_IBD>8'])

# Calculate weighted centrality using betweenness centrality (or any other centrality measure you prefer)
weighted_centrality = nx.betweenness_centrality(G, weight='weight')

# Calculate the weighted strength of each node (sum of edge weights incident to the node)
strength = {}
for node in G.nodes():
    strength[node] = sum(d['weight'] for _, d in G[node].items())

# Convert centrality and strength to DataFrame for easy export
centrality_df = pd.DataFrame(list(weighted_centrality.items()), columns=['Node', 'Weighted Centrality'])
strength_df = pd.DataFrame(list(strength.items()), columns=['Node', 'Weighted Strength'])

# Export both DataFrames to CSV
centrality_df.to_csv("weighted_centrality.csv", index=False)
strength_df.to_csv("node_strength.csv", index=False)

print("CSV files exported: 'weighted_centrality.csv' and 'node_strength.csv'")


In [ ]:
import pandas as pd

# Load the node_strength.csv
strength_df = pd.read_csv("node_strength.csv")

# Load the second CSV with columns: name, Date updated, Degree, Sex, Yhap-group
info_df = pd.read_csv("/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/KNC_v12_16cM_fracgp0.7.node.csv")  # replace with the actual file name

# Rename 'name' column to 'Node' to match the column name in strength_df
info_df.rename(columns={'name': 'Node'}, inplace=True)

# Merge the two DataFrames on 'Node'
merged_df = pd.merge(strength_df, info_df, on='Node', how='left')

# Export the merged DataFrame to a new CSV
merged_df.to_csv("merged_node_data.csv", index=False)

print("CSV files merged and exported to 'merged_node_data.csv'")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the CSV file
df = pd.read_csv(
    '/mnt/archgen/users/xiaowen/Kamenice/1024/IBD/merged_node_data.csv',
    names=['Name', 'Strength','Phase', 'Degree', 'Sex', 'Yhap'], skiprows=1
)

# Function to calculate and plot CDF for a given phase and sex
def plot_cdf(df, phase, sex, label, color, linestyle='-'):
    df_subset = df[(df['Phase'] == phase) & (df['Sex'] == sex)]
    degree_sequence = df_subset['Strength']
    degree_count = degree_sequence.value_counts().sort_index()
    frequencies = degree_count.sort_index(ascending=False)
    cumulative_freq = np.cumsum(frequencies)
    total_nodes = len(df_subset)
    if total_nodes == 0:
        return  # skip if no nodes
    P_k_greater_x = cumulative_freq / total_nodes
    degree_values = frequencies.index.to_numpy()
    P_k_greater_x = P_k_greater_x.to_numpy()
    plt.plot(degree_values, P_k_greater_x, marker='o', linestyle=linestyle, color=color, label=label)

# Start the plot
figure = plt.figure(figsize=(8, 7), dpi=600)


# Plot for all individuals regardless of phase/sex
degree_sequence = df['Strength']
degree_count = degree_sequence.value_counts().sort_index()
frequencies = degree_count.sort_index(ascending=False)
cumulative_freq = np.cumsum(frequencies)
total_nodes = len(df)
P_k_greater_x = cumulative_freq / total_nodes
degree_values = frequencies.index.to_numpy()
P_k_greater_x = P_k_greater_x.to_numpy()
#plt.plot(degree_values, P_k_greater_x, marker='o', linestyle='-', color='grey', label='All Individuals')

# Define colors for phases
phase_colors = {'KNC-MLBA': '#999900', 'KNC-EIA': '#4CC26C', 'KNC-DIA': '#365C8D'}

# Plot for each phase and sex
for phase, color in phase_colors.items():
    plot_cdf(df, phase, 'M', label=f'{phase} - Male', color=color, linestyle='-')
    plot_cdf(df, phase, 'F', label=f'{phase} - Female', color=color, linestyle='--')

# Final plot details
plt.title('CDF of Strength Distribution by Phase and Sex',fontsize=16)
plt.xlabel('strength centrality (w)',fontsize=16)
plt.ylabel('P(w > x)',fontsize=16)
plt.legend(fontsize=16)
#plt.grid(True)
plt.tick_params(axis='both', labelsize=12)
plt.tight_layout()
plt.show()
figure.savefig("CDF_strength.svg", format='svg', bbox_inches='tight')


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# Create a scalar mappable for the colorbar
norm = Normalize(vmin=0, vmax=0.533)
cmap = plt.cm.viridis_r
sm = ScalarMappable(cmap=cmap, norm=norm)

# Create a figure only for the colorbar
fig, ax = plt.subplots(figsize=(1.0, 4.0),dpi=600)
fig.subplots_adjust(left=0.3, right=0.6)

# Add colorbar as a legend
cbar = plt.colorbar(sm, cax=ax, ticks=[0, 0.15, 0.3, 0.533])
cbar.set_label("Betweeness centrality", rotation=90, labelpad=10, fontsize=16)
cbar.ax.tick_params(labelsize=14)

# Hide all axes
ax.tick_params(size=0)
ax.set_frame_on(False)

# Display
plt.show()
fig.savefig("betweeness_colorbar.svg", format='svg', bbox_inches='tight')